In [ ]:
# Colab session setup — runs only on Colab; harmless when running locally.
# Mounts Drive, clones the repo (idempotent), installs the package via uv into system Python.
try:
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    if os.path.exists('/content/MyBookRec'):
        !cd /content/MyBookRec && git pull -q
    else:
        !git clone -q https://github.com/zali78690/MyBookRec.git /content/MyBookRec

    %cd /content/MyBookRec
    !pip install -q uv
    !uv pip install --system -q -e .
    print("Colab setup complete")
except ImportError:
    print("Not on Colab — skipping setup")

## The goal
Train the two-tower model end to end.

Inputs (all precomputed in earlier steps):
- `train_user_features.npy` — `(n_users, 779)` matrix indexed by `user_idx`
- `book_embeddings.npy` + `genre_matrix.npy` + `num_pages_normalized.npy` — concatenated at load time to form a `(n_books, 395)` item feature matrix
- `TrainingPairsDataset` (step 14) — emits `(user_idx, pos_book_idx, neg_book_idxs)` per call with fresh negatives

Loop per epoch:
1. For each batch from the train DataLoader, look up user + item features by index, run them through the two towers, compute BCE loss on the `[1, 0, 0, 0, 0]` labels, backprop, optimizer step.
2. Compute validation loss on the val split (same Dataset structure, no backprop).
3. Save a checkpoint if val loss improved; track best.

Hit Rate@10 is deferred to step 18 (eval module). Val loss is the proxy here.

Run on GPU (Colab/Kaggle T4) for real training. On CPU, do 1-2 epochs for sanity only — a full run takes hours.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from mybookrec import DATA_DIR
from mybookrec.features.training_pairs import TrainingPairsDataset
from mybookrec.model.towers import TwoTowerModel

# On Colab, redirect DATA_DIR to the Drive layout: MyBookRec/data/transformed/...
# Local run: this is a no-op because the path doesn't exist.
_colab_data = Path('/content/drive/MyDrive/MyBookRec/data')
if _colab_data.exists():
    DATA_DIR = _colab_data
print(f"DATA_DIR: {DATA_DIR}")
assert (DATA_DIR / "transformed" / "book_id_to_index.json").exists(), f"Data not found at {DATA_DIR}"

In [ ]:
# Hyperparameters tuned for a single T4 (16GB) on Colab.
HIDDEN_DIMS = (512, 256, 128)   # 3-layer MLP; last entry is the shared embedding dim
DROPOUT = 0.1
BATCH_SIZE = 4096               # T4 has plenty of headroom; larger batch = better GPU utilization
N_NEGATIVES = 4
LR = 1e-3                       # Adam is robust to this across 1k-8k batch sizes
N_EPOCHS = 10                   # 10 is a good first real run; bump to 30-50 if val loss is still falling
EARLY_STOPPING_PATIENCE = 5     # more patience because val loss has noise from per-call negative sampling
CHECKPOINT_DIR = DATA_DIR.parent / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Load and concatenate item features into a single (n_books, 395) matrix.
# Order: [description_embedding (384) | genre (10) | normalized_pages (1)].
# This order is the contract — must match at inference time.
transformed = DATA_DIR / "transformed"
book_embeddings = np.load(transformed / "book_embeddings.npy").astype(np.float32)
genre_matrix = np.load(transformed / "genre_matrix.npy").astype(np.float32)
pages_vec = np.load(transformed / "num_pages_normalized.npy").astype(np.float32)

item_features_np = np.concatenate(
    [book_embeddings, genre_matrix, pages_vec.reshape(-1, 1)], axis=1
).astype(np.float32)
del book_embeddings, genre_matrix, pages_vec  # free intermediate copies

user_features_np = np.load(transformed / "train_user_features.npy").astype(np.float32)

print(f"item_features: {item_features_np.shape}  ({item_features_np.nbytes / 1e9:.2f} GB)")
print(f"user_features: {user_features_np.shape}  ({user_features_np.nbytes / 1e9:.2f} GB)")

In [ ]:
# Move feature matrices to the training device.
# These tensors are read-only lookup tables for the whole training run — no grad needed.
user_features = torch.from_numpy(user_features_np).to(device)
item_features = torch.from_numpy(item_features_np).to(device)
del user_features_np, item_features_np

user_input_dim = user_features.shape[1]
item_input_dim = item_features.shape[1]
print(f"user_input_dim={user_input_dim}  item_input_dim={item_input_dim}")

In [ ]:
# Build train and validation datasets. Both use fresh negative sampling.
train_dataset = TrainingPairsDataset(n_negatives=N_NEGATIVES, data_split="train")
val_dataset = TrainingPairsDataset(n_negatives=N_NEGATIVES, data_split="validation", rng_seed=0)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False
)

print(f"Train batches: {len(train_loader):,}  (~{len(train_dataset):,} positives)")
print(f"Val batches:   {len(val_loader):,}  (~{len(val_dataset):,} positives)")

In [ ]:
# Instantiate model, optimizer, loss.
model = TwoTowerModel(
    user_input_dim=user_input_dim,
    item_input_dim=item_input_dim,
    hidden_dims=HIDDEN_DIMS,
    dropout=DROPOUT,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.BCEWithLogitsLoss()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"Trainable params: {n_params:,}")

In [ ]:
# Smoke test: one forward pass on a fresh batch confirms shapes line up before we start training.
model.eval()
with torch.no_grad():
    u_idx, p_idx, n_idx = next(iter(train_loader))
    item_idx = torch.cat([p_idx.unsqueeze(1), n_idx], dim=1).to(device)        # (B, 5)
    user_feat = user_features[u_idx.to(device)]                                 # (B, 779)
    item_feat = item_features[item_idx]                                         # (B, 5, 395)
    similarity = model(user_feat, item_feat)                                    # (B, 5)
    labels = torch.zeros_like(similarity)
    labels[:, 0] = 1.0
    loss = loss_fn(similarity, labels)
    print(f"similarity shape: {tuple(similarity.shape)}")
    print(f"similarity range: [{similarity.min().item():.3f}, {similarity.max().item():.3f}]")
    print(f"smoke-test loss:  {loss.item():.4f}")
model.train()

In [ ]:
def run_epoch(loader: DataLoader, train: bool) -> float:
    """Iterate one pass over the loader. Returns mean loss across all examples."""
    model.train(train)
    total_loss = 0.0
    total_examples = 0

    grad_ctx = torch.enable_grad() if train else torch.no_grad()
    with grad_ctx:
        for u_idx, p_idx, n_idx in loader:
            u_idx = u_idx.to(device)
            item_idx = torch.cat([p_idx.unsqueeze(1), n_idx], dim=1).to(device)
            user_feat = user_features[u_idx]
            item_feat = item_features[item_idx]
            similarity = model(user_feat, item_feat)

            labels = torch.zeros_like(similarity)
            labels[:, 0] = 1.0
            loss = loss_fn(similarity, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            batch_examples = similarity.numel()
            total_loss += loss.item() * batch_examples
            total_examples += batch_examples

    return total_loss / total_examples

In [ ]:
# Main training loop.
best_val_loss = float("inf")
epochs_since_improvement = 0
history: list[dict] = []

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    elapsed = time.time() - t0

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "sec": elapsed})
    print(f"Epoch {epoch:3d}  train={train_loss:.4f}  val={val_loss:.4f}  ({elapsed:.0f}s)")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_since_improvement = 0
        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "val_loss": val_loss,
                "config": {
                    "hidden_dims": HIDDEN_DIMS,
                    "dropout": DROPOUT,
                    "user_input_dim": user_input_dim,
                    "item_input_dim": item_input_dim,
                },
            },
            CHECKPOINT_DIR / "two_tower_best.pt",
        )
        print(f"  ↳ new best, checkpoint saved")
    else:
        epochs_since_improvement += 1
        if epochs_since_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"  ↳ no improvement for {EARLY_STOPPING_PATIENCE} epochs, early stopping")
            break

with open(CHECKPOINT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)
print(f"\nBest val loss: {best_val_loss:.4f}")